In [ ]:
import os
import time
import cv2
import numpy as np
import tensorflow as tf
from matplotlib import pyplot as plt
from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetPowerUsage
)
from voxelmorph.tf.layers import SpatialTransformer
from voxelmorph.tf.losses import Grad

MODEL_PATH = '../training/models/vxm2d_model_ez.h5'
DATA_DIR = '../../workspace/calibration_dataset'
GPU_INDEX = 0
NUM_SAMPLES = 20
NUM_RUNS = 5
BASELINE_SAMPLES = 50

def load_pair(idx):
    arr = np.load(os.path.join(DATA_DIR, f'calib_pair_{idx}.npy'), allow_pickle=True)
    moving = arr[0][..., 0].astype(np.float32)
    fixed = arr[1][..., 0].astype(np.float32)
    return moving, fixed

def init_models():
    model = tf.keras.models.load_model(
        MODEL_PATH,
        custom_objects={'SpatialTransformer': SpatialTransformer},
        compile=False
    )
    model.compile(
        optimizer='adam',
        loss=['mse', Grad('l2')],
        loss_weights=[1.0, 0.01]
    )
    extractor = tf.keras.Model(
        inputs=model.inputs,
        outputs=model.get_layer('leaky_re_lu_118').output
    )
    return model, extractor

def measure_baseline(handle):
    readings = []
    for _ in range(BASELINE_SAMPLES):
        readings.append(nvmlDeviceGetPowerUsage(handle) / 1000)
        time.sleep(0.05)
    return sum(readings) / len(readings)

def warp_image(model, moving, fixed):
    _, flow_batch = model.predict([moving[None], fixed[None]], verbose=0)
    flow = flow_batch[0]
    h, w = flow.shape[:2]
    gx, gy = np.meshgrid(np.arange(w), np.arange(h))
    mx = (gx + flow[...,1]).astype(np.float32)
    my = (gy + flow[...,0]).astype(np.float32)
    warped = cv2.remap(moving, mx, my, cv2.INTER_LINEAR, cv2.BORDER_REFLECT)
    return flow, warped

def compute_metrics(warped, fixed, flow):
    mse = np.mean((warped - fixed)**2)
    dx = flow[1:] - flow[:-1]
    dy = flow[:,1:] - flow[:,:-1]
    grad_norm = (np.sum(dx*dx) + np.sum(dy*dy)) / np.prod(flow.shape)
    total_loss = mse + 0.01 * grad_norm
    return mse, grad_norm, total_loss

model, extractor = init_models()

lat_means = []
lat_stds = []

nvmlInit()
handle = nvmlDeviceGetHandleByIndex(GPU_INDEX)

for run in range(1, NUM_RUNS+1):
    latencies = []
    powers = []
    mses = []
    grads = []
    losses = []

    baseline = measure_baseline(handle)
    print(f'Baseline power: {baseline:.2f} W')

    for i in range(NUM_SAMPLES):
        moving, fixed = load_pair(i)
        p0 = nvmlDeviceGetPowerUsage(handle) / 1000
        t0 = time.perf_counter()

        with tf.device('/GPU:0'):
            _ = extractor.predict([moving[None], fixed[None]], verbose=0)

        t1 = time.perf_counter()
        p1 = nvmlDeviceGetPowerUsage(handle) / 1000

        latencies.append((t1 - t0) * 1e3)
        powers.append(((p0 + p1)/2) - baseline)

        flow, warped = warp_image(model, moving, fixed)
        mse, grad_norm, loss = compute_metrics(warped, fixed, flow)
        mses.append(mse)
        grads.append(grad_norm)
        losses.append(loss)

        if i == 0:
            first_flow, first_mov, first_fix, first_warp = flow, moving, fixed, warped

    arr = np.array(latencies)
    lat_means.append(arr.mean())
    lat_stds.append(arr.std())

    print(f'Run {run}/{NUM_RUNS} — '
          f'Latency: {arr.mean():.2f}±{arr.std():.2f} ms | '
          f'MSE: {np.mean(mses):.6e} | '
          f'Grad: {np.mean(grads):.6e} | '
          f'Loss: {np.mean(losses):.6e}')

nvmlShutdown()

lat_means = np.array(lat_means)
lat_stds = np.array(lat_stds)

os.makedirs('results/gpu', exist_ok=True)
np.save('results/gpu/latency_mean_gpu.npy', lat_means)
np.save('results/gpu/latency_sd_gpu.npy', lat_stds)
np.save('results/gpu/loss_results.npy', {'mse': np.mean(mses), 'grad': np.mean(grads), 'loss': np.mean(losses)})

print(f'\nOverall mean latency: {lat_means.mean():.2f} ms')
print(f'Overall std latency:  {lat_stds.mean():.2f} ms')

u, v = first_flow[..., 0], first_flow[...,1]
step = 8
ys, xs = np.arange(0, first_warp.shape[0], step), np.arange(0, first_warp.shape[1], step)
X, Y = np.meshgrid(xs, ys)
U, V = u[::step, ::step], v[::step, ::step]

plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.imshow(first_mov, cmap='gray')
plt.title('Moving Image')
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(first_fix, cmap='gray')
plt.title('Fixed Image')
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(first_warp, cmap='gray')
plt.quiver(X, Y, U, V, angles='xy', scale_units='xy', scale=1,
           width=0.003, headwidth=3, headlength=4, alpha=0.8, color='r')
plt.title('Warped + Flow')
plt.axis('off')

plt.tight_layout()
plt.show()